# Procédure d'obfuscation AloePri — Qwen/Qwen3-8B

**Objectif** — Reproduire de bout en bout la procédure d'obfuscation AloePri
sur `Qwen/Qwen3-8B` :
1. **Section 1** — construction des matrices d'obfuscation (permutation de
   vocabulaire Π, bruit d'embedding, facteurs d'attention/FFN, matrices clés
   P̂/Q̂) ;
2. **Section 2** — transformation du modèle sur Modal (streaming, ~16 Go),
   vérification bit-à-bit et récupération des clés ;
3. **Section 3** — export et service serverless du modèle obfusqué.

Les sections 4-6 (attaque ISA, évaluation, bilan) sont ajoutées par des
tâches ultérieures : ce notebook est le socle exécutable (sections 0-3).

**Modèle de menace** — L'opérateur du serveur d'inférence ne doit pas pouvoir
reconstruire la représentation interne du modèle à partir des seuls poids
obfusqués (permutation de vocabulaire, bruit α_e/α_h, facteurs orthogonaux,
matrices clés). Les clés restent **exclusivement côté client** : elles ne sont
jamais montées par le service.

**Références** — Papier : [AloePri — arXiv:2603.01499](https://arxiv.org/pdf/2603.01499)
(Algorithme 1, §5.2.2, §5.4) · Spec : `docs/superpowers/specs/2026-08-24-aloepri-notebook-design.md`
· Code : package `aloepri/` (port T1-T3, 40/40 tests) + `modal_app.py`.

---

## 0. Setup

In [ ]:
import os, sys
# bootstrap : racine du worktree sur sys.path. Le kernel nbconvert démarre
# dans le dossier du notebook (notebooks/) ; on remonte jusqu'à trouver le
# package aloepri/ pour que les imports des cellules 1.5 et 2.1 fonctionnent
# quel que soit le point de lancement (nbconvert, Jupyter racine ou notebooks/).
for _ in range(6):
    if os.path.isdir(os.path.join(os.getcwd(), "aloepri")):
        break
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import numpy as np
import torch

# Constantes de la procédure (valeurs du plan — à utiliser telles quelles)
MODEL = "Qwen/Qwen3-8B"
SEED = 0
ALPHA_E = 0.3
BETA = 8
THINK_ID = 151667

# Drapeau : False → exécution locale rapide (sections 0-1 + branches "sauté") ;
# True → exécute réellement les cellules Modal (transform, verify, deploy,
# health check). Passer à True uniquement avec une CLI Modal authentifiée.
RUN_HEAVY = False

In [ ]:
import os

print(f"racine      {os.getcwd()}")
print(f"torch      {torch.__version__}")
print(f"numpy      {np.__version__}")
assert isinstance(RUN_HEAVY, bool), "RUN_HEAVY doit être un booléen"
print(f"RUN_HEAVY  {RUN_HEAVY} (booléen ✓)")

modal_cli = os.path.expanduser("~/modal-venv/bin/modal")
print(f"Modal CLI  {modal_cli} -> " + ("accessible ✓" if os.path.exists(modal_cli)
                                       else "introuvable (cellules RUN_HEAVY indisponibles)"))

### Comment exécuter ce notebook

- **Environnement local** — venv `.venv/` (numpy 2.5, torch 2.13 CPU,
  transformers 5.15, jupyter 1.1.1). Lancer Jupyter **depuis la racine du
  worktree** (`jupyter notebook notebooks/aloepri_procedure.ipynb`) : les
  imports `aloepri` (cellules 1.5 et 2.1) en dépendent.
- **`RUN_HEAVY = False` (défaut)** — exécution locale rapide : sections 0-1
  complètes ; les cellules Modal des sections 2-3 affichent
  `[RUN_HEAVY=False] … sauté` (aucun appel Modal, aucune authentification).
- **`RUN_HEAVY = True`** — exécute réellement les cellules Modal : transform
  (~16 Go, ~30-60 min CPU), récupération + suppression des clés, verify
  (~30 min), deploy + health check. Prérequis : CLI `~/modal-venv/bin/modal`
  authentifiée (`~/modal-venv/bin/modal token new`) ; les volumes Modal
  (`obfuscator-models`, `obfuscator-keys`) sont créés automatiquement.
- **Coûts** — transform ≈ 1 h CPU ; verify ≈ 30 min CPU ; serve : GPU L4
  scale-to-zero (facturation à la seconde). Les clés sont téléchargées dans
  `artifacts/obfuscation_keys.json` (gitignoré).

## 1. Matrices d'obfuscation

Cellules autonomes (numpy/torch, petites échelles) qui illustrent chaque
facteur de la procédure ; la transformation réelle (section 2) applique les
mêmes constructions aux tenseurs du modèle.

### 1.1 Permutation de vocabulaire Π

Permutation déterministe (seed `SEED`) des `V` lignes d'embedding / colonnes
de `lm_head`. `unperm` est l'inverse exacte : Π·Π⁻¹ = Id.

In [ ]:
import numpy as np
V = 1000
rng = np.random.default_rng(SEED)
perm = rng.permutation(V)
unperm = np.empty_like(perm); unperm[perm] = np.arange(V)
# vérification : Π·Π⁻¹ = Id
assert (perm[unperm] == np.arange(V)).all() and (unperm[perm] == np.arange(V)).all()
print(f"Π construite : {V} tokens, inverse exacte ✓")

### 1.2 Bruit d'embedding (α_e, α_h)

Bruit gaussien relatif aux poids : σ(bruit) = α_e · σ(W) (embedding/head) ;
le même principe s'applique aux activations cachées avec α_h.

In [ ]:
w = torch.randn(64, 128)
noise = ALPHA_E * torch.randn_like(w) * w.std()   # rapport bruit/signal = α_e
assert abs(noise.std() / w.std() - ALPHA_E) < 0.1
print(f"σ(bruit)/σ(poids) ≈ {noise.std()/w.std():.2f} ≈ α_e ✓")

### 1.3 Facteurs d'attention

- **R̂** — rotation RoPE par paires `(i, i+d/2)` (layout `half` de
  `rotate_half`, Qwen2/Qwen3) ;
- **Ĥ** — facteur diagonal — **désactivé sur Qwen3** (`rope_scaling=off`) :
  les normes de tête `q_norm`/`k_norm` ne commutent pas avec un facteur
  diagonal ;
- **Ẑ** — permutation de blocs de largeur `d_head/β` ;
- **Û_vo** — matrice orthogonale (décomposition QR).

In [ ]:
d_head = 32
beta = BETA  # β du papier (8) ; l'exemple ci-dessous illustre β_ex = 3 blocs
# Ẑ : permutation de blocs de largeur d_head//3 (exemple β_ex=3). d_head=32
# n'est pas multiple de 3 → les lignes restantes sont laissées à l'identité
# pour que Ẑ soit une permutation complète (orthogonale).
blk = [1, 2, 0]
w = d_head // len(blk)
Z = torch.zeros(d_head, d_head)
for j, src in enumerate(blk):
    Z[j*w:(j+1)*w, src*w:(src+1)*w] = torch.eye(w)
for r in range(len(blk) * w, d_head):
    Z[r, r] = 1.0
assert torch.allclose(Z @ Z.T, torch.eye(d_head)), "Ẑ doit être une permutation (orthogonale)"
# Û_vo : orthogonale via QR
U, _ = torch.linalg.qr(torch.randn(d_head, d_head))
assert torch.allclose(U.T @ U, torch.eye(d_head), atol=1e-5), "Û_vo doit être orthogonale"
print("R̂/Ẑ/Û_vo : facteurs orthogonaux vérifiés ✓")

### 1.4 Facteurs FFN

Permutation des neurones de la couche intermédiaire + scaling multiplicatif
`exp(N(0, 0.1))` (bruit relatif ~10 % par neurone).

In [ ]:
h = 64
rng = np.random.default_rng(SEED + 7)
neu = rng.permutation(h)
scale = torch.exp(0.1 * torch.randn(h))
assert len(set(neu.tolist())) == h
print(f"FFN : permutation de {h} neurones + scalings ∈ [exp(±0.1·N)] ✓")

### 1.5 Matrices clés P̂/Q̂ — aperçu (Algorithme 1)

Implémentation complète à venir en **Section 6** ; démo de l'API
`aloepri.key_matrix` (Algorithme 1, arXiv:2603.01499 §5.4) : `P̂` de forme
`(d, d+2h)` et `Q̂` de forme `(d+2h, d)` tels que **P̂·Q̂ = I_d**.

In [ ]:
from aloepri.key_matrix import init_key_matrix, key_mat_gen, inv_key_mat_gen
import numpy.random as npr
base = init_key_matrix(d=64, h=8, lam=0.3, rng=npr.default_rng(SEED))
P = key_mat_gen(base); Q = inv_key_mat_gen(base)
err = float(np.abs(P @ Q - np.eye(64)).max())
assert err < 1e-10, f"P̂·Q̂=I attendu, erreur max {err}"
print(f"P̂ ({P.shape}) · Q̂ ({Q.shape}) = I, erreur max {err:.2e} ✓")

## 2. Obfuscation du modèle (Modal)

### 2.1 Vérification d'architecture

`aloepri.check_arch` valide les hypothèses du POC sur `Qwen/Qwen3-8B`
(config seule — aucun poids téléchargé) : GQA (`num_key_value_heads` divise
`num_attention_heads`), `head_dim` pair (RoPE), biais q/k/v, **`q_norm` /
`k_norm` (Qwen3) → `rope_scaling=off`**, layout RoPE `half` (rotate_half),
weight tying, cohérence du vocabulaire.

In [ ]:
from aloepri.check_arch import check  # API réelle : check() (le brief disait check_arch)
import contextlib, io

buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    rc = check(MODEL)  # télécharge uniquement la config, jamais les poids
print(buf.getvalue(), end="")
assert rc == 0, "check() a signalé des hypothèses en échec"
assert "toutes les hypothèses sont satisfaites" in buf.getvalue(), \
    "le rapport doit conclure : toutes les hypothèses sont satisfaites"
print("✓ toutes les hypothèses sont satisfaites — la transformation peut être lancée")

### 2.2 Transformation sur Modal

Transformation réelle en streaming (mémoire-léger, ~16 Go de poids, ~30-60
min CPU) : écrit le modèle obfusqué sur le volume `obfuscator-models`
(`qwen3-8b-obf`) et les clés sur `obfuscator-keys`. Cellule conditionnée par
`RUN_HEAVY` — appel CLI robuste (équivalent de
`modal.Function.lookup("obfuscator-aloepri","transform").remote(seed=SEED,
alpha_e=ALPHA_E, beta=BETA)`) :

```bash
~/modal-venv/bin/modal run modal_app.py::transform --seed 0 --alpha-e 0.3 --beta 8
```

In [ ]:
if RUN_HEAVY:
    !~/modal-venv/bin/modal run modal_app.py::transform --seed 0 --alpha-e 0.3 --beta 8
else:
    print("[RUN_HEAVY=False] transform() sauté — résultat attendu : keys_sha256 + out_subdir='qwen3-8b-obf'")

### 2.3 Vérification bit-à-bit + récupération des clés

1. Télécharger les clés du volume `obfuscator-keys` vers
   `artifacts/obfuscation_keys.json` (gitignoré) ;
2. **Supprimer le volume clés** — posture : les clés ne restent jamais sur
   Modal (`verify()` régénère les clés par seed, il n'en a pas besoin) ;
3. `verify()` avec **les mêmes hyperparamètres que `transform()`**
   (`--seed 0 --alpha-e 0.3 --beta 8`) — échantillons de lignes embed/head +
   couches complètes ; assert : 0 écart bit-à-bit.

In [ ]:
if RUN_HEAVY:
    import os
    os.makedirs("artifacts", exist_ok=True)
    !~/modal-venv/bin/modal volume get obfuscator-keys /obfuscation_keys.json artifacts/obfuscation_keys.json
    # Posture de sécurité : les clés restent côté client (artifacts/) — le
    # volume clés est supprimé de Modal (verify() régénère les clés par seed).
    !~/modal-venv/bin/modal volume delete obfuscator-keys -y
    verify_out = !~/modal-venv/bin/modal run modal_app.py::verify --seed 0 --alpha-e 0.3 --beta 8
    print(verify_out.s)
    assert "[OK]" in verify_out.s, "verify() doit rapporter 0 écart bit-à-bit sur les échantillons"
    print("✓ verify() : 0 écart bit-à-bit sur les échantillons")
else:
    print("[RUN_HEAVY=False] récupération des clés + verify() sauté — attendu : 0 écart bit-à-bit")

## 3. Export et service (Modal)

### 3.1 Volume modèle

Le modèle obfusqué est servi depuis le volume `obfuscator-models`,
sous-répertoire `/qwen3-8b-obf` (écrit par `transform()`, section 2.2).
Inspection manuelle (authentification Modal requise) :
`~/modal-venv/bin/modal volume ls obfuscator-models /qwen3-8b-obf`.

In [ ]:
if RUN_HEAVY:
    !~/modal-venv/bin/modal deploy modal_app.py
else:
    print("[RUN_HEAVY=False] deploy sauté — attendu : https://mauceri--obfuscator-aloepri-serve.modal.run")

### 3.2 Cold start

Le service est scale-to-zero : le premier appel après une période d'inactivité
peut renvoyer `503` (boot du conteneur, ~1-2 min). La boucle de health check
ci-dessous attend jusqu'à 5 min avant d'abandonner.

In [ ]:
URL = "https://mauceri--obfuscator-aloepri-serve.modal.run"

if RUN_HEAVY:
    import os, time, requests
    api_key = open(os.path.expanduser("~/.aloepri-api-key")).read().strip() \
        if os.path.exists(os.path.expanduser("~/.aloepri-api-key")) else None
    headers = {"Authorization": f"Bearer {api_key}"} if api_key else {}
    for _ in range(60):
        try:
            if requests.get(f"{URL}/health", headers=headers, timeout=10).status_code == 200:
                print("service prêt ✓"); break
        except requests.RequestException:
            time.sleep(5)
    else:
        raise SystemExit("service injoignable après 5 min")
else:
    print("[RUN_HEAVY=False] health check sauté — attendu : service prêt ✓")